# Exploratory Data Analysis (EDA) and Model Training

## 1. Load and Inspect Data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Load datasets
aq_df = pd.read_csv('../data/air_quality_data.csv')
health_df = pd.read_csv('../data/health_data.csv')

print("Air Quality Data:")
print(aq_df.head())
print("
Health Data:")
print(health_df.head())

## 2. Data Cleaning and Preprocessing

In [ ]:
# Handle missing values
aq_df.dropna(inplace=True)
health_df.dropna(inplace=True)

# Convert 'Date' column to datetime
aq_df['Date'] = pd.to_datetime(aq_df['Date'])
health_df['Date'] = pd.to_datetime(health_df['Date'])

# Merge datasets
df = pd.merge(aq_df, health_df, on=['City', 'Date'])

print("Merged Dataframe:")
print(df.head())

## 3. AQI Trend Analysis

In [ ]:
# AQI trend over time
fig = px.line(df, x='Date', y='AQI', color='City', title='AQI Trend Over Time')
fig.show()

# Save as static image
plt.figure(figsize=(12, 6))
sns.lineplot(x='Date', y='AQI', hue='City', data=df)
plt.title('AQI Trend Over Time')
plt.savefig('../visuals/aqi_trends.png')
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation between pollutants and health issues
corr_cols = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI', 'Asthma_Cases', 'Bronchitis_Cases', 'Respiratory_Admissions']
correlation_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.savefig('../visuals/health_correlation.png')
plt.show()

## 5. AQI Prediction Model

In [ ]:
# Feature Engineering
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

# Define features and target
features = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'Year', 'Month', 'Day']
target = 'AQI'

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a RandomForest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'R-squared: {r2_score(y_test, y_pred)}')

# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred)
plt.xlabel('Actual AQI')
plt.ylabel('Predicted AQI')
plt.title('Actual vs. Predicted AQI')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.savefig('../visuals/regression_plot.png')
plt.show()

## 6. Save the Model

In [ ]:
# Save the trained model
joblib.dump(model, '../models/aqi_model.joblib')
print("Model saved to ../models/aqi_model.joblib")